# Monster Hunting: a step-by-step anomaly notebook

This notebook is intentionally simple and explicit.

We will:

1. load the **actual training tables**;
2. inspect what each table contains;
3. create one clean activity table from video and LMS events;
4. run each monster rule separately;
5. explain what each result means;
6. avoid claiming findings when the required table is unavailable.

The available row-level training files are:

- `la_mmh_training_video_activity_by_view_fact.csv`
- `la_mmh_training_lms_item_activity_by_session_fact.csv`
- `la_mmh_training_person_dim.csv`
- `la_mmh_training_person_module_instance_fact.csv`

The other files in the folder are data dictionaries, not row-level data.

## Important definitions

An anomaly result is a **candidate**, not automatically a confirmed problem.

- **Phantom:** repeated activity records.
- **Ghost:** active in at least 3 of the first 4 Autumn weeks, then below 10% of the earlier weekly average. Requires the real calendar table.
- **Zombie:** activity continues for more than 3 hours, with no gap longer than 15 minutes.
- **Vampire:** more active days during 00:00–05:00 than during 05:00–24:00, with more than 21 night-active days.
- **Undead Student:** activity after the student's deregistration date.
- **Doppelgänger:** students with identical activity profiles. The notebook uses the simple video-only rule matching the challenge-style code.
- **Shape-shifter:** programme/module change. Requires the real module dimension table.

In [2]:
from pathlib import Path
import itertools
import math
import re

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# Change only this path if your folder is elsewhere.
DATA_DIR = Path(r"C:\Users\cvill\Downloads\Training (1)\Training")

print("Folder exists:", DATA_DIR.exists())
print("Folder:", DATA_DIR)


Folder exists: True
Folder: C:\Users\cvill\Downloads\Training (1)\Training


## 1. Load the real training tables

We use explicit filenames here. This prevents the notebook from accidentally loading the similarly named data-dictionary files.

In [3]:
FILE_MAP = {
    "video_activity": "la_mmh_training_video_activity_by_view_fact.csv",
    "lms_activity": "la_mmh_training_lms_item_activity_by_session_fact.csv",
    "person": "la_mmh_training_person_dim.csv",
    "person_module": "la_mmh_training_person_module_instance_fact.csv",
}

tables = {}

for table_name, filename in FILE_MAP.items():
    path = DATA_DIR / filename

    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    df = pd.read_csv(path)
    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]
    tables[table_name] = df

    print(f"{table_name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print("  columns:", list(df.columns))


video_activity: 1,145,358 rows x 7 columns
  columns: ['PVT_ID', 'P_ID', 'VIDEO_ID', 'DATE', 'TIMESTAMP', 'TOTAL_MINUTES_DELIVERED', 'VIEWING_TYPE']
lms_activity: 7,857,424 rows x 8 columns
  columns: ['PIT_ID', 'DATE', 'P_ID', 'LMS_ITEM_ID', 'TIMESTAMP', 'LOGIN_ID', 'INTERACTION_COUNT', 'TOTAL_SECONDS_BEFORE_NEXT_INTERACTION']
person: 14,951 rows x 3 columns
  columns: ['P_ID', 'registration_date', 'deregistration_date']
person_module: 100,611 rows x 4 columns
  columns: ['PMI_ID', 'P_ID', 'MODULE_INSTANCE_ID', 'MODULE_REGISTRATION_STATUS']


### What to check

The two activity tables should contain `P_ID`, `DATE`, and `TIMESTAMP`.

The person table should contain `P_ID` and `deregistration_date`.

If the columns shown here are `field_name`, `data_type`, and `example_data`, you loaded a data dictionary instead of the training file.

In [4]:
for name, df in tables.items():
    print(f"\n{name}")
    display(df.head(3))



video_activity


,PVT_ID,P_ID,VIDEO_ID,DATE,TIMESTAMP,TOTAL_MINUTES_DELIVERED,VIEWING_TYPE
0,PVT0000000001,P00015925,V000018648,2024-01-30,2024-01-30T17:04:47.000Z,21.8851,Interactive Desktop On Demand
1,PVT0000000002,P00011628,V000020715,2023-12-10,2023-12-10T13:52:15.000Z,0.6533,Interactive Desktop On Demand
2,PVT0000000003,P00011432,V000001230,2023-11-01,2023-11-01T16:22:58.000Z,2.5063,Interactive Desktop On Demand



lms_activity


,PIT_ID,DATE,P_ID,LMS_ITEM_ID,TIMESTAMP,LOGIN_ID,INTERACTION_COUNT,TOTAL_SECONDS_BEFORE_NEXT_INTERACTION
0,PIT0009356404,2024-01-04,P00013902,2206942531,2024-01-04T15:45:27.722Z,L00537558,2,3
1,PIT0009356406,2024-03-06,P00015594,1951450793,2024-03-06T14:55:04.942Z,L02125951,2,14
2,PIT0009356410,2024-04-10,P00011450,2253629406,2024-04-10T14:09:48.805Z,L01525121,8,1489



person


,P_ID,registration_date,deregistration_date
0,P00001187,2023-09-01,NaN
1,P00006491,2023-09-21,NaN
2,P00007122,2023-09-05,NaN



person_module


,PMI_ID,P_ID,MODULE_INSTANCE_ID,MODULE_REGISTRATION_STATUS
0,PMI0153397,P00012615,I2301485,Mandatory
1,PMI0062532,P00006447,I2301485,Mandatory
2,PMI0090410,P00016549,I2302486,Mandatory


## 2. Standardise the activity data

In [5]:
def clean_column_name(column):
    column = str(column).replace("\ufeff", "").replace("\xa0", " ")
    column = column.strip().upper()
    column = re.sub(r"[^A-Z0-9]+", "_", column)
    return column.strip("_")


def prepare_columns(df):
    out = df.copy()
    out.columns = [clean_column_name(c) for c in out.columns]
    return out


def standardise_activity(df, source):
    out = prepare_columns(df)
    out["_SOURCE"] = source

    if "P_ID" not in out.columns:
        raise KeyError(f"{source}: P_ID is missing")
    if "TIMESTAMP" not in out.columns:
        raise KeyError(f"{source}: TIMESTAMP is missing")
    if "DATE" not in out.columns:
        raise KeyError(f"{source}: DATE is missing")

    out["_P_ID"] = out["P_ID"].astype("string")
    out["_TIMESTAMP"] = pd.to_datetime(
        out["TIMESTAMP"], errors="coerce", utc=True
    )
    out["_DATE"] = pd.to_datetime(
        out["DATE"], errors="coerce", utc=True
    ).dt.normalize()

    return out


video = standardise_activity(tables["video_activity"], "video")
lms = standardise_activity(tables["lms_activity"], "lms")

activity = pd.concat([video, lms], ignore_index=True, sort=False)

print("Video rows:", f"{len(video):,}")
print("LMS rows:", f"{len(lms):,}")
print("Combined activity rows:", f"{len(activity):,}")

display(activity[["_SOURCE", "_P_ID", "_TIMESTAMP", "_DATE"]].head())


Video rows: 1,145,358
LMS rows: 7,857,424
Combined activity rows: 9,002,782


,_SOURCE,_P_ID,_TIMESTAMP,_DATE
0,video,P00015925,2024-01-30 17:04:47+00:00,2024-01-30 00:00:00+00:00
1,video,P00011628,2023-12-10 13:52:15+00:00,2023-12-10 00:00:00+00:00
2,video,P00011432,2023-11-01 16:22:58+00:00,2023-11-01 00:00:00+00:00
3,video,P00001646,2023-10-18 11:08:57+00:00,2023-10-18 00:00:00+00:00
4,video,P00004377,2024-03-14 01:54:06+00:00,2024-03-14 00:00:00+00:00


This creates one common activity table.

- `_SOURCE` says whether a row came from video or LMS.
- `_P_ID` is the common student ID.
- `_TIMESTAMP` is the event time.
- `_DATE` is the event date.

No anomaly has been detected yet.

## 3. Basic data-quality checks

In [ ]:
quality = pd.DataFrame({
    "rows": [len(video), len(lms), len(activity)],
    "missing_P_ID": [
        video["_P_ID"].isna().sum(),
        lms["_P_ID"].isna().sum(),
        activity["_P_ID"].isna().sum(),
    ],
    "missing_timestamp": [
        video["_TIMESTAMP"].isna().sum(),
        lms["_TIMESTAMP"].isna().sum(),
        activity["_TIMESTAMP"].isna().sum(),
    ],
    "missing_date": [
        video["_DATE"].isna().sum(),
        lms["_DATE"].isna().sum(),
        activity["_DATE"].isna().sum(),
    ],
}, index=["video", "lms", "combined"])

display(quality)
print("Unique students in activity:", activity["_P_ID"].nunique())
print("Activity date range:", activity["_DATE"].min(), "to", activity["_DATE"].max())


## 4. Undead Student

Rule: keep activity whose date is **strictly after** the student's deregistration date.

Activity on the deregistration date itself is not flagged.

In [6]:
people = prepare_columns(tables["person"])
people["P_ID"] = people["P_ID"].astype("string")
people["DEREGISTRATION_DATE"] = pd.to_datetime(
    people["DEREGISTRATION_DATE"], errors="coerce", utc=True
).dt.normalize()

# Check whether person_dim has one row per student.
print("Person rows:", len(people))
print("Unique P_IDs:", people["P_ID"].nunique())
print("Duplicate P_ID rows:", people["P_ID"].duplicated().sum())

undead = activity.copy()
undead["_ACTIVITY_DATE"] = undead["_TIMESTAMP"].dt.normalize()

undead = undead.merge(
    people[["P_ID", "DEREGISTRATION_DATE"]],
    left_on="_P_ID",
    right_on="P_ID",
    how="left"
)

undead = undead[
    undead["DEREGISTRATION_DATE"].notna()
    & undead["_ACTIVITY_DATE"].notna()
    & (undead["_ACTIVITY_DATE"] > undead["DEREGISTRATION_DATE"])
].copy()

undead_summary = (
    undead.groupby(["_P_ID", "DEREGISTRATION_DATE"], as_index=False)
    .agg(
        SOURCES=("_SOURCE", lambda s: ", ".join(sorted(s.unique()))),
        POST_DEREG_ACTIVITY_ROWS=("_TIMESTAMP", "size"),
        FIRST_POST_DEREG_ACTIVITY=("_TIMESTAMP", "min"),
        LAST_POST_DEREG_ACTIVITY=("_TIMESTAMP", "max"),
    )
    .sort_values("POST_DEREG_ACTIVITY_ROWS", ascending=False)
)

print(f"Undead candidates: {len(undead_summary):,}")
display(undead_summary.head(20))


Person rows: 14951
Unique P_IDs: 14951
Duplicate P_ID rows: 0
Undead candidates: 92


,_P_ID,DEREGISTRATION_DATE,SOURCES,POST_DEREG_ACTIVITY_ROWS,FIRST_POST_DEREG_ACTIVITY,LAST_POST_DEREG_ACTIVITY
79,P00019372,2023-11-15 00:00:00+00:00,"lms, video",134,2023-11-20 10:05:11.565000+00:00,2024-01-08 02:36:48.011000+00:00
10,P00002127,2024-01-05 00:00:00+00:00,"lms, video",102,2024-01-09 15:20:02.358000+00:00,2024-02-05 15:26:08.337000+00:00
24,P00004134,2024-02-04 00:00:00+00:00,video,64,2024-02-08 10:50:40+00:00,2024-03-19 14:02:09+00:00
57,P00014003,2024-01-03 00:00:00+00:00,lms,64,2024-01-09 11:04:18.133000+00:00,2024-02-26 15:12:53.979000+00:00
91,P00022151,2024-03-14 00:00:00+00:00,video,55,2024-03-30 07:52:58+00:00,2024-04-25 09:06:54+00:00
61,P00015029,2024-04-19 00:00:00+00:00,video,45,2024-04-20 10:24:17+00:00,2024-04-25 12:26:12+00:00
90,P00022023,2024-02-04 00:00:00+00:00,video,42,2024-02-05 11:24:13+00:00,2024-03-12 18:38:13+00:00
33,P00005688,2024-01-29 00:00:00+00:00,video,37,2024-01-30 15:27:07+00:00,2024-03-22 08:55:19+00:00
22,P00004036,2024-01-11 00:00:00+00:00,"lms, video",36,2024-01-12 05:17:57+00:00,2024-02-09 10:46:04.128000+00:00
37,P00006185,2024-01-26 00:00:00+00:00,video,34,2024-02-08 11:45:23+00:00,2024-03-16 19:51:25+00:00


Interpretation: each row in `undead_summary` is a student with at least one activity date after deregistration.

Before treating this as a real anomaly, check duplicate `P_ID`s and possible date/time-zone issues.

## 5. Phantom

We run two clearly separated checks:

1. **Exact duplicate rows:** compare every original field, including technical IDs.
2. **Repeated activity patterns:** ignore technical IDs and compare behavioural fields.

The first check respects the fact that IDs may be important. The second is exploratory and must not be called an exact duplicate check.

In [7]:
def exact_duplicate_groups(df, source):
    # Keep every original column, including PVT_ID or PIT_ID.
    cols = [c for c in df.columns if not c.startswith("_")]

    duplicated = df[df.duplicated(subset=cols, keep=False)].copy()

    if duplicated.empty:
        return pd.DataFrame()

    result = (
        duplicated.groupby(cols, dropna=False, sort=False)
        .size()
        .reset_index(name="DUPLICATE_COUNT")
        .sort_values("DUPLICATE_COUNT", ascending=False)
    )
    result.insert(0, "SOURCE", source)
    return result


phantom_exact = pd.concat([
    exact_duplicate_groups(video, "video"),
    exact_duplicate_groups(lms, "lms"),
], ignore_index=True, sort=False)

print(f"Exact duplicate groups: {len(phantom_exact):,}")
display(phantom_exact.head(20))


Exact duplicate groups: 70,286


,SOURCE,PVT_ID,P_ID,VIDEO_ID,DATE,TIMESTAMP,TOTAL_MINUTES_DELIVERED,VIEWING_TYPE,DUPLICATE_COUNT,PIT_ID,LMS_ITEM_ID,LOGIN_ID,INTERACTION_COUNT,TOTAL_SECONDS_BEFORE_NEXT_INTERACTION
0,video,PVT0000000097,P00001840,V000012498,2024-05-08,2024-05-08T20:38:58.000Z,5.9987,Interactive Desktop On Demand,2,NaN,NaN,NaN,NaN,NaN
1,video,PVT0000958647,P00008215,V000020735,2023-11-28,2023-11-28T19:08:44.000Z,54.2441,Interactive Desktop On Demand,2,NaN,NaN,NaN,NaN,NaN
2,video,PVT0000961635,P00001840,V000020253,2023-10-24,2023-10-24T21:47:22.000Z,31.3666,Interactive Desktop On Demand,2,NaN,NaN,NaN,NaN,NaN
3,video,PVT0000961185,P00021588,V000014190,2024-05-27,2024-05-27T14:16:53.000Z,3.2694,Interactive Desktop On Demand,2,NaN,NaN,NaN,NaN,NaN
4,video,PVT0000961141,P00008215,V000018308,2024-01-07,2024-01-07T10:55:10.000Z,39.5112,Interactive Desktop On Demand,2,NaN,NaN,NaN,NaN,NaN
5,video,PVT0000961099,P00008975,V000018039,2024-04-01,2024-04-01T20:15:52.000Z,0.0653,Interactive Desktop On Demand,2,NaN,NaN,NaN,NaN,NaN
6,video,PVT0000960938,P00003116,V000026358,2024-01-16,2024-01-16T20:40:04.000Z,21.5816,Interactive Desktop On Demand,2,NaN,NaN,NaN,NaN,NaN
7,video,PVT0000960779,P00012076,V000016755,2024-01-14,2024-01-14T15:20:11.000Z,3.7825,Interactive Desktop On Demand,2,NaN,NaN,NaN,NaN,NaN
8,video,PVT0000960613,P00017379,V000005011,2023-11-24,2023-11-24T13:05:43.000Z,28.9173,Interactive Desktop On Demand,2,NaN,NaN,NaN,NaN,NaN
9,video,PVT0000960582,P00014989,V000028903,2024-05-02,2024-05-02T12:08:39.000Z,32.0097,Interactive Desktop On Demand,2,NaN,NaN,NaN,NaN,NaN


In [ ]:
def repeated_activity_patterns(df, source):
    ignored = {"PVT_ID", "PIT_ID", "PMI_ID", "LOGIN_ID", "LOG_IN_ID"}
    cols = [
        c for c in df.columns
        if c not in ignored and not c.startswith("_")
    ]

    duplicated = df[df.duplicated(subset=cols, keep=False)].copy()

    if duplicated.empty:
        return pd.DataFrame()

    result = (
        duplicated.groupby(cols, dropna=False, sort=False)
        .size()
        .reset_index(name="REPEATED_PATTERN_COUNT")
        .sort_values("REPEATED_PATTERN_COUNT", ascending=False)
    )
    result.insert(0, "SOURCE", source)
    return result


phantom_patterns = pd.concat([
    repeated_activity_patterns(video, "video"),
    repeated_activity_patterns(lms, "lms"),
], ignore_index=True, sort=False)

print(f"Repeated behavioural patterns: {len(phantom_patterns):,}")
display(phantom_patterns.head(20))


The two counts answer different questions.

- `phantom_exact`: the complete original row repeats.
- `phantom_patterns`: the activity description repeats after technical IDs are ignored.

Neither result alone proves that the repeated record is erroneous.

## 6. Doppelgänger

This version follows the simple challenge-style rule:

- use the video table;
- define each event as `VIDEO_ID + DATE + TIMESTAMP`;
- sort each student's complete event list;
- identify identical student profiles.

The output counts groups and students separately.

In [6]:
video_d = video.dropna(
    subset=["_P_ID", "VIDEO_ID", "_DATE", "_TIMESTAMP"]
).copy()

video_d["EVENT_SIGNATURE"] = (
    video_d["VIDEO_ID"].astype("string")
    + "|"
    + video_d["_DATE"].dt.strftime("%Y-%m-%d")
    + "|"
    + video_d["_TIMESTAMP"].astype("string")
)

student_profiles = (
    video_d.groupby("_P_ID")["EVENT_SIGNATURE"]
    .apply(lambda values: tuple(sorted(values.tolist())))
    .reset_index(name="PROFILE")
)

shared_profiles = student_profiles[
    student_profiles.duplicated(subset=["PROFILE"], keep=False)
].copy()

doppel_groups = (
    shared_profiles.groupby("PROFILE")["_P_ID"]
    .agg(list)
    .reset_index(name="P_IDS")
)

doppel_groups["NUMBER_OF_STUDENTS"] = doppel_groups["P_IDS"].str.len()
doppel_groups["NUMBER_OF_PAIRS"] = doppel_groups["NUMBER_OF_STUDENTS"].apply(
    lambda n: math.comb(n, 2)
)

print("Shared profile groups:", len(doppel_groups))
print("Students involved:", shared_profiles["_P_ID"].nunique())
print("All possible student pairs:", doppel_groups["NUMBER_OF_PAIRS"].sum())

display(doppel_groups.head(20))


Shared profile groups: 30
Students involved: 60
All possible student pairs: 30


,PROFILE,P_IDS,NUMBER_OF_STUDENTS,NUMBER_OF_PAIRS
0,(V00000001|2023-10-30|2023-10-30 14:06:59+00:0...,"[P00010770, P00907330]",2,1
1,(V00000050|2023-10-11|2023-10-11 11:29:00+00:0...,"[P00012287, P00365560]",2,1
2,(V00000059|2024-02-05|2024-02-05 10:18:51+00:0...,"[P00003382, P00922502]",2,1
3,(V00000078|2024-02-03|2024-02-03 16:50:00+00:0...,"[P00021322, P00948045]",2,1
4,(V00000081|2023-12-05|2023-12-05 13:37:20+00:0...,"[P00002959, P00336066]",2,1
5,(V00000097|2024-01-29|2024-01-29 12:12:17+00:0...,"[P00019901, P00624727]",2,1
6,(V000001060|2024-04-17|2024-04-17 14:28:54+00:...,"[P00007309, P00219495]",2,1
7,(V000001062|2024-04-24|2024-04-24 22:18:28+00:...,"[P00017482, P00944765]",2,1
8,(V000001083|2024-02-06|2024-02-06 15:56:19+00:...,"[P00016841, P00408220]",2,1
9,(V00000108|2024-04-16|2024-04-16 03:31:42+00:0...,"[P00000512, P00116388]",2,1


In [ ]:
# Submission-style output: one P_ID per row, each student only once.
doppel_submission = (
    shared_profiles[["_P_ID"]]
    .drop_duplicates()
    .rename(columns={"_P_ID": "P_ID"})
    .sort_values("P_ID")
)

display(doppel_submission.head(20))

# Optional export
doppel_submission.to_csv("doppelganger_pids.csv", index=False)
print("Saved: doppelganger_pids.csv")


## 7. Zombie

Rule: for a student, activity timestamps form a continuous run when consecutive events are no more than 15 minutes apart. A run is a Zombie candidate when it lasts more than 3 hours.

This uses both video and LMS activity.

In [8]:
z = activity.dropna(subset=["_P_ID", "_TIMESTAMP"]).copy()
z = z[["_P_ID", "_TIMESTAMP", "_SOURCE"]].sort_values(["_P_ID", "_TIMESTAMP"])

z["GAP_MINUTES"] = (
    z.groupby("_P_ID")["_TIMESTAMP"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

# A new run starts after a gap greater than 15 minutes.
z["RUN_ID"] = (
    z["GAP_MINUTES"].isna()
    | (z["GAP_MINUTES"] > 15)
).groupby(z["_P_ID"]).cumsum()

zombie_runs = (
    z.groupby(["_P_ID", "RUN_ID"], as_index=False)
    .agg(
        START=("_TIMESTAMP", "min"),
        END=("_TIMESTAMP", "max"),
        ACTIVITY_ROWS=("_TIMESTAMP", "size"),
        SOURCES=("_SOURCE", lambda s: ", ".join(sorted(s.unique()))),
    )
)

zombie_runs["DURATION_HOURS"] = (
    zombie_runs["END"] - zombie_runs["START"]
).dt.total_seconds() / 3600

zombie_candidates = zombie_runs[
    zombie_runs["DURATION_HOURS"] > 3
].sort_values("DURATION_HOURS", ascending=False)

print(f"Zombie runs: {len(zombie_candidates):,}")
print(f"Zombie students: {zombie_candidates['_P_ID'].nunique():,}")
display(zombie_candidates.head(20))


Zombie runs: 36
Zombie students: 25


,_P_ID,RUN_ID,START,END,ACTIVITY_ROWS,SOURCES,DURATION_HOURS
2943924,P00020056,721,2024-01-03 23:44:44.699000+00:00,2024-01-04 04:52:19+00:00,78,"lms, video",5.126195
666516,P00004100,948,2024-03-19 00:00:42+00:00,2024-03-19 04:59:55+00:00,97,lms,4.986944
1096158,P00006867,1724,2024-03-22 00:00:07+00:00,2024-03-22 04:54:15+00:00,76,"lms, video",4.902222
666502,P00004100,934,2024-03-18 00:00:39+00:00,2024-03-18 04:48:44+00:00,89,lms,4.801389
131905,P00000806,117,2023-10-20 00:00:21+00:00,2023-10-20 04:29:02+00:00,49,lms,4.478056
1095177,P00006867,743,2023-12-22 00:05:22+00:00,2023-12-22 04:22:05+00:00,75,lms,4.278611
2959466,P00021355,703,2024-01-31 00:56:39+00:00,2024-01-31 04:58:37+00:00,50,"lms, video",4.032778
2954146,P00021098,224,2023-10-27 00:52:25+00:00,2023-10-27 04:49:21+00:00,44,lms,3.948889
1427854,P00008877,606,2024-01-16 00:04:50+00:00,2024-01-16 03:50:20+00:00,47,"lms, video",3.758333
124123,P00000775,226,2023-10-26 00:00:18+00:00,2023-10-26 03:44:39+00:00,57,lms,3.739167


## 8. Vampire

Rule:

- night is 00:00 inclusive to 05:00 exclusive;
- day is 05:00 inclusive to midnight;
- retain students with more than 21 night-active days;
- retain students with more night-active days than day-active days.

In [ ]:
v = activity.dropna(subset=["_P_ID", "_TIMESTAMP"]).copy()
v["ACTIVITY_DATE"] = v["_TIMESTAMP"].dt.normalize()
v["HOUR"] = v["_TIMESTAMP"].dt.hour
v["PERIOD"] = np.where(v["HOUR"] < 5, "night", "day")

days = (
    v.groupby(["_P_ID", "PERIOD"])["ACTIVITY_DATE"]
    .nunique()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ["night", "day"]:
    if col not in days.columns:
        days[col] = 0

vampire_candidates = days[
    (days["night"] > 21)
    & (days["night"] > days["day"])
].copy()

vampire_candidates = vampire_candidates.sort_values(
    ["night", "day"], ascending=[False, True]
)

print(f"Vampire students: {len(vampire_candidates):,}")
display(vampire_candidates.head(20))


## 9. Ghost

Ghost requires the actual calendar dimension with at least:

- `DATE`;
- `WEEK_NUMBER`;
- `TERM`.

In this dataset folder, `la_mmh_calendar_dim.csv` is a data dictionary, not row-level calendar data. Therefore we do not invent Autumn weeks or run this rule.

In [ ]:
ghost_result = {
    "status": "not_evaluated",
    "reason": (
        "The folder contains the calendar data dictionary but not the "
        "row-level calendar dimension. Autumn weeks cannot be identified reliably."
    ),
    "candidates": pd.DataFrame(),
}

print("Ghost:", ghost_result["status"])
print(ghost_result["reason"])


## 10. Shape-shifter

Shape-shifter requires the actual module dimension, especially a programme/department field such as `DEPARTMENT_ID`.

The available `person_module_instance_fact` table tells us which module instances a person registered for, but without the real module dimension we cannot determine whether the programme changed. We therefore report this as unavailable rather than guessing.

In [7]:
shape_shifter_result = {
    "status": "not_evaluated",
    "reason": (
        "The actual module_instance_dim table is unavailable. "
        "Programme changes cannot be inferred reliably from the available tables."
    ),
    "candidates": pd.DataFrame(),
}

print("Shape-shifter:", shape_shifter_result["status"])
print(shape_shifter_result["reason"])


Shape-shifter: not_evaluated
The actual module_instance_dim table is unavailable. Programme changes cannot be inferred reliably from the available tables.


## 11. Final result summary

This table records what was actually evaluated, using the definitions in this notebook.

In [ ]:
summary = pd.DataFrame([
    {
        "MONSTER": "Phantom exact",
        "STATUS": "evaluated",
        "COUNT": len(phantom_exact),
        "UNIT": "duplicate groups",
        "DEFINITION": "complete original activity rows repeat",
    },
    {
        "MONSTER": "Phantom behavioural",
        "STATUS": "evaluated",
        "COUNT": len(phantom_patterns),
        "UNIT": "repeated groups",
        "DEFINITION": "activity repeats after technical IDs are ignored",
    },
    {
        "MONSTER": "Doppelganger",
        "STATUS": "evaluated",
        "COUNT": len(doppel_groups),
        "UNIT": "shared video profiles",
        "DEFINITION": "identical VIDEO_ID + DATE + TIMESTAMP profiles",
    },
    {
        "MONSTER": "Zombie",
        "STATUS": "evaluated",
        "COUNT": len(zombie_candidates),
        "UNIT": "continuous runs",
        "DEFINITION": "more than 3 hours with no gap over 15 minutes",
    },
    {
        "MONSTER": "Vampire",
        "STATUS": "evaluated",
        "COUNT": len(vampire_candidates),
        "UNIT": "students",
        "DEFINITION": "more than 21 night days and more night than day days",
    },
    {
        "MONSTER": "Undead Student",
        "STATUS": "evaluated",
        "COUNT": len(undead_summary),
        "UNIT": "students",
        "DEFINITION": "activity strictly after deregistration",
    },
    {
        "MONSTER": "Ghost",
        "STATUS": "not evaluated",
        "COUNT": np.nan,
        "UNIT": "students",
        "DEFINITION": "requires row-level calendar data",
    },
    {
        "MONSTER": "Shape-shifter",
        "STATUS": "not evaluated",
        "COUNT": np.nan,
        "UNIT": "students",
        "DEFINITION": "requires row-level module dimension data",
    },
])

display(summary)


## What this notebook does not do

It does not prove that a student cheated or that a row is wrong.

It produces transparent candidate lists that can later be used to:

- inspect the source records;
- validate the anomaly definitions;
- create labelled training data;
- build a predictive model for unseen data.

Every candidate should be validated against the underlying event records before being treated as a confirmed anomaly.